# Tydzień 10: Prognozowanie szeregów czasowych – CO₂ na Mauna Loa 🌍📈

| | |
|---|---|
| **Rozdział książki** | 7. *Time Series Forecasting* |
| **Czas zajęć** | 90 minut |
| **Dane** | `data/mauna_loa_co2.csv` – miesięczne stężenie CO₂ w atmosferze 1958–2021 (plik jest już w repozytorium) |
| **Nowe funkcje PyCaret** | moduł `pycaret.time_series`: `setup(fh=...)`, `check_stats()`, `compare_models()`, `predict_model(fh=...)` |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. wiedzieć, czym jest **szereg czasowy** (*time series*) i **prognozowanie** (*forecasting*),
2. umieć rozłożyć szereg na **trend**, **sezonowość** i **resztę** (dekompozycja),
3. rozumieć, czym jest **stacjonarność** i dlaczego w szeregach czasowych **nie wolno mieszać danych** przy walidacji,
4. umieć porównać modele prognostyczne w PyCaret i zrobić prognozę CO₂ na kolejne 3 lata.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–20 min | 📚 Teoria: trend, sezonowość, wygładzanie wykładnicze |
| 20–70 min | 👣 Krok po kroku |
| 70–85 min | 🧑‍💻 Ćwiczenie: kiedy przekroczymy 430 ppm? – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Szereg czasowy i prognozowanie ⏱️
**Szereg czasowy** to pomiary tej samej wielkości w **regularnych odstępach czasu**: temperatura co godzinę,
sprzedaż co tydzień, stężenie CO₂ co miesiąc. **Prognozowanie** to przewidywanie przyszłych wartości szeregu.

Historia danych: w 1958 r. Charles David Keeling zaczął mierzyć CO₂ w obserwatorium **Mauna Loa** na Hawajach (z dala od miast i fabryk).
Wykres tych pomiarów, tzw. **krzywa Keelinga**, to jeden z najważniejszych dowodów na rosnące stężenie CO₂ w atmosferze.

## Trzy składniki szeregu 🧩
Wyobraź sobie spacer z psem pod górę 🐕⛰️:
- **Trend** (*trend*) – ogólny kierunek: idziecie **pod górę** (długoterminowy wzrost CO₂).
- **Sezonowość** (*seasonality*) – regularny, powtarzający się wzór: pies biega **zygzakiem** w lewo i prawo
  (CO₂ co roku spada latem, gdy rośliny na półkuli północnej rosną i pochłaniają CO₂, a rośnie zimą – „oddech planety” 🌱).
- **Reszta / szum** (*residual / noise*) – drobne, losowe odchylenia (pies zatrzymał się powąchać kwiatek).

## Stacjonarność (*Stationarity*) ⚖️
Szereg jest **stacjonarny**, gdy jego „charakter” (średnia, zmienność) **nie zmienia się w czasie** – jak poziom wody
w wannie z zamkniętym kranem. Szereg CO₂ z rosnącym trendem **nie jest stacjonarny**. Niektóre modele tego wymagają,
dlatego sprawdzamy to testami statystycznymi (ADF i KPSS) – PyCaret zrobi to jednym poleceniem.

## Walidacja w szeregach czasowych – bez zaglądania w przyszłość 🔮🚫
W regresji mogliśmy losowo mieszać wiersze. Tu **nie wolno**! Model nie może uczyć się na danych z 2015 r., żeby „przewidzieć” rok 2010.
PyCaret używa **rosnącego okna** (*expanding window*): trenuje na danych do 2009 → sprawdza na kolejnych 3 latach,
trenuje do 2012 → sprawdza dalej, itd. Zawsze **przeszłość → przyszłość**.

**Horyzont prognozy** (*forecast horizon*, `fh`) – na ile kroków do przodu prognozujemy (u nas 36 miesięcy = 3 lata).

## Wygładzanie wykładnicze (*Exponential Smoothing*) 🌡️
Prognoza to **średnia ważona** przeszłych obserwacji, w której **nowsze pomiary mają większą wagę** – tak jak przewidując jutrzejszą
pogodę bardziej ufasz wczorajszej niż tej sprzed roku. Wersja Holta-Wintersa śledzi osobno **poziom** (ℓ), **trend** (b) i **sezonowość** (s):

$$\hat{y}_{t+h} = \ell_t + h\,b_t + s_{t+h-m}$$

## Metryka MAPE 📏
**MAPE** (*Mean Absolute Percentage Error*) – średni błąd **w procentach**. MAPE = 0,0011 oznacza, że prognoza myli się średnio o **0,11%**.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Tabele i wykresy
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Narzędzia statsmodels: dekompozycja szeregu i wykres autokorelacji
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf

In [ ]:
# Moduł SZEREGÓW CZASOWYCH PyCaret
from pycaret.time_series import *

### Krok 2: Wczytujemy dane 📥

In [ ]:
# index_col='datetime' – kolumna z datą staje się indeksem (etykietami wierszy); parse_dates=True – zamień tekst na daty
data = pd.read_csv('../data/mauna_loa_co2.csv', index_col='datetime', parse_dates=True)
data.head()

In [ ]:
# 766 miesięcznych pomiarów od marca 1958 do grudnia 2021, bez braków danych
data.info()

### Krok 3: Analiza szeregu czasowego 🔍

In [ ]:
# Wykres liniowy całego szeregu
data.plot(figsize=(10, 5), title='Miesięczne stężenie CO₂ (ppm)')
plt.show()

🔎 Widać wyraźny **trend wzrostowy** (z ok. 315 do ponad 415 ppm) oraz regularne „ząbki” – **sezonowość**.
ppm (*parts per million*) = liczba cząsteczek CO₂ na milion cząsteczek powietrza.

In [ ]:
# Przybliżenie: tylko lata 2018–2021 – sezonowość widać dużo lepiej
data['2018':].plot(figsize=(10, 4), marker='o')
plt.show()

In [ ]:
# Dekompozycja: period=12 – sezon trwa 12 miesięcy. Wynik ma składowe: trend, seasonal, resid
result = seasonal_decompose(data['CO2'], period=12)

In [ ]:
# Wykres 4 paneli: dane, trend, sezonowość, reszta
result.plot()
plt.show()

In [ ]:
# Składowa sezonowa dla 12 miesięcy jednego roku: o ile ppm dany miesiąc jest powyżej/poniżej trendu
result.seasonal['2020'].round(2)

🔎 Stężenie jest najwyższe w **maju** (ok. +3 ppm ponad trend), a najniższe we **wrześniu–październiku** (ok. −3 ppm) –
po letnim okresie wzrostu roślin. To właśnie „oddech planety”.

In [ ]:
# Autokorelacja (ACF): jak bardzo wartość jest podobna do wartości sprzed 1, 2, 3… miesięcy
plot_acf(data['CO2'], lags=36)
plt.show()

🔎 Wszystkie słupki są wysokie i **bardzo powoli** maleją – to typowy obraz szeregu z silnym trendem (każdy miesiąc jest
podobny do poprzednich, bo „wszystko rośnie”). Słaba sezonowość ginie w cieniu trendu.

### Krok 4: Inicjalizacja środowiska – `setup()` ⚙️
Najpierw drobna zmiana formatu: zamiast konkretnych dni (np. 1958-03-31) używamy **miesięcy** (1958-03).
Tak PyCaret najlepiej rozumie dane miesięczne, a prognozy też będą miały format miesięczny.

In [ ]:
# to_period('M') – zamiana dat na okresy miesięczne (M = month)
data.index = data.index.to_period('M')
data.tail(3)

In [ ]:
# fh=36 – horyzont prognozy: 36 miesięcy. Ostatnie 36 miesięcy trafią do zbioru testowego.
ts = setup(data, fh=36, session_id=6501)

📖 **Najważniejsze wiersze tabeli:**
- `Transformed train set shape` (730, 1) i `Transformed test set shape` (36, 1) – ostatnie 3 lata to „egzamin”,
- `Fold Generator` = **ExpandingWindowSplitter** – rosnące okno (bez zaglądania w przyszłość!), `Fold Number` = 3,
- `Seasonality Present` = True, `Primary Seasonality` = **12** – PyCaret **sam wykrył** roczną sezonowość. 🎉

### Krok 5: Testy statystyczne – czy szereg jest stacjonarny? 🧪

In [ ]:
# Testy stacjonarności ADF i KPSS
check_stats(test='stationarity')

🔎 W wierszach `Stationarity` / `Trend Stationarity` oba testy mówią **False** – szereg **nie jest stacjonarny**
(co widzieliśmy na wykresie, ale dobrze to potwierdzić testem).

### Krok 6: Porównanie modeli prognostycznych 🏁
PyCaret ma ok. 30 modeli prognostycznych – od bardzo prostych po uczenie maszynowe. Wszystkie razem liczyłyby się kilka minut,
dlatego wybierzemy reprezentatywną grupę:

| ID | Model | Idea w jednym zdaniu |
|---|---|---|
| `naive` | Naiwny | „jutro będzie tak jak dziś” |
| `snaive` | Sezonowy naiwny | „w tym maju będzie tak jak w zeszłym maju” |
| `polytrend` | Trend wielomianowy | dopasowuje gładką krzywą trendu |
| `arima` | ARIMA | klasyczny model statystyczny |
| `exp_smooth` | Wygładzanie wykładnicze | średnia ważona z większą wagą dla nowych danych |
| `theta` | Theta | prosty i często zaskakująco skuteczny |
| `lr_cds_dt` | Regresja liniowa (ML) | zamienia prognozowanie w zadanie regresji na opóźnionych wartościach |

In [ ]:
# Ranking według MAPE (średni błąd procentowy); ⏳ ok. 30–60 sekund
best = compare_models(include=['naive', 'snaive', 'polytrend', 'arima', 'exp_smooth', 'theta', 'lr_cds_dt'], sort='MAPE')

🔎 Zwycięża **wygładzanie wykładnicze** (`exp_smooth`) z MAPE ≈ **0,0011** – czyli średni błąd to zaledwie **0,11%**!
Proste modele „naiwne” są dużo gorsze, bo nie uwzględniają trendu.

### Krok 7: Budujemy model wygładzania wykładniczego

In [ ]:
# Model exp_smooth oceniony walidacją krzyżową z rosnącym oknem (3 części)
model = create_model('exp_smooth')

📖 Kolumna `cutoff` mówi, **do którego miesiąca** model był trenowany w danej części (np. 2009-12), a metryki dotyczą
kolejnych 36 miesięcy po tej dacie.

In [ ]:
# Prognoza na zbiorze testowym (2019–2021) na tle danych (interaktywny wykres – możesz przybliżać)
plot_model(model, plot='forecast')

In [ ]:
# Diagnostyka reszt (błędów modelu): wykres reszt, histogram, wykres Q-Q, ACF/PACF
plot_model(model, plot='diagnostics', fig_kwargs={'height': 900, 'width': 1000})

📖 **Jak czytać diagnostykę?** Dobry model ma reszty (błędy) przypominające **losowy szum**:
- **histogram** reszt w kształcie „dzwonu” (rozkład normalny),
- punkty na **wykresie Q-Q** blisko prostej linii,
- słupki **ACF/PACF** reszt blisko zera – błędy nie zawierają już żadnego wzoru, który model mógłby jeszcze wykorzystać.

U nas wszystkie te warunki są spełnione – model wyciągnął z danych prawie całą informację. 👍

### Krok 8: Finalizacja i prognoza na lata 2022–2024 🔮

In [ ]:
# Trenujemy model na WSZYSTKICH danych 1958–2021
final_model = finalize_model(model)

In [ ]:
# Prognoza na 36 miesięcy w przód (styczeń 2022 – grudzień 2024)
pred = predict_model(final_model, fh=36)
pred.head()

In [ ]:
# Wykres: ostatnie 8 lat danych (96 miesięcy) + prognoza
ax = data[-96:].plot(figsize=(10, 5))
pred.plot(ax=ax, color='orange')
ax.legend(['Pomiary', 'Prognoza'])
plt.show()

🔎 Prognoza zachowuje i **trend**, i **sezonowość**. A jak wypada w porównaniu z rzeczywistością?
Rzeczywiste pomiary z maja 2024 r. wyniosły ok. **427 ppm**. Sprawdźmy, ile przewidział nasz model:

In [ ]:
# Prognoza dla maja 2024 r. (loc wybiera wiersz po etykiecie indeksu)
pred.loc['2024-05']

🔎 Model przewidział ok. **426,7 ppm** – pomylił się zaledwie o ok. 0,3 ppm, prognozując ponad 2 lata do przodu! 🎯

In [ ]:
# Zapis modelu prognostycznego
import os
os.makedirs('../modele', exist_ok=True)
save_model(final_model, '../modele/exp_smooth_model');

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Kiedy przekroczymy 430 ppm?** 🌡️

- **a)** Zrób prognozę modelem `final_model` na **60 miesięcy** (5 lat) do przodu i zapisz ją w zmiennej `pred60`.
- **b)** Znajdź **pierwszy miesiąc**, w którym prognozowane stężenie (`y_pred`) przekracza **430 ppm**.
- **c)** (dla chętnych) Zbuduj model `'theta'` i narysuj jego prognozę na zbiorze testowym (`plot_model(..., plot='forecast')`).
  Czy wygląda lepiej, czy gorzej niż wygładzanie wykładnicze?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `predict_model(final_model, fh=60)`
- b) Filtrowanie: `pred60[pred60['y_pred'] > 430]`, a potem `.head(1)` – pierwszy wiersz.
- c) `create_model('theta')`, potem `plot_model(theta, plot='forecast')`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Prognoza z przedziałem niepewności
Prognoza to nigdy jedna pewna liczba. Lepiej podać **przedział**, w którym przyszła wartość znajdzie się z dużym prawdopodobieństwem.

- **a)** Spróbuj uzyskać przedziały z modelu `final_model` (wygładzanie wykładnicze):
  `predict_model(final_model, fh=36, return_pred_int=True)`. Co widzisz w kolumnach `lower` i `upper`?
- **b)** Zbuduj, sfinalizuj i wykorzystaj model **ARIMA** (`'arima'`) – on potrafi liczyć przedziały.
- **c)** Narysuj ostatnie 8 lat pomiarów, prognozę ARIMA i **zacieniowany** przedział niepewności.
  Jak zmienia się szerokość przedziału im dalej w przyszłość?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- `return_pred_int=True` dodaje kolumny `lower` (dolna granica) i `upper` (górna granica) – domyślnie przedział 95%.
- `NaN` oznacza „brak wartości” – nie każdy model umie policzyć przedziały.
- Rysowanie: zamień indeksy-miesiące na daty (`.to_timestamp()`), a potem użyj matplotlib:
  `plt.plot(...)` dla linii i `plt.fill_between(x, dolna, górna, alpha=0.3)` dla zacieniowanego pasa.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Nowy szereg – pasażerowie linii lotniczych ✈️
Zastosuj cały „przepis” do innego, słynnego szeregu: miesięcznej liczby pasażerów linii lotniczych w latach 1949–1960
(w tysiącach), zbiór `airline` z PyCaret.

- **a)** Wczytaj dane i narysuj szereg. Jakie widzisz trend i sezonowość? Czym ta sezonowość różni się od sezonowości CO₂?
- **b)** Przygotuj środowisko z horyzontem **12 miesięcy** (`session_id=123`) i porównaj modele
  `'naive'`, `'snaive'`, `'exp_smooth'`, `'arima'`, `'theta'`, `'lr_cds_dt'` według MAPE.
- **c)** Sfinalizuj najlepszy model, zrób prognozę na rok **1961** i narysuj ją razem z danymi.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- `get_data('airline', verbose=False)` zwraca od razu szereg z miesięcznym indeksem – nie trzeba nic zamieniać.
- `compare_models(...)` zwraca najlepszy model; potem `finalize_model(...)` i `predict_model(..., fh=12)`.
- Wykres: tak jak w kroku 8 (`dane.plot()`, a potem `prognoza.plot(ax=ax)`).

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **Szereg czasowy** = pomiary w regularnych odstępach czasu; składa się z **trendu**, **sezonowości** i **reszty**.
- `seasonal_decompose()` rozkłada szereg na składowe; `plot_acf()` pokazuje autokorelację.
- Szereg z trendem **nie jest stacjonarny** – sprawdzamy to `check_stats()` (testy ADF i KPSS).
- Walidacja w szeregach czasowych używa **rosnącego okna** – nigdy nie uczymy się na przyszłości.
- `setup(data, fh=36)` → `compare_models()` → `create_model()` → `finalize_model()` → `predict_model(fh=...)` – ten sam przepis co zawsze!
- **Wygładzanie wykładnicze** dało prognozę z błędem ok. 0,1%.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| szereg czasowy | time series | pomiary w regularnych odstępach czasu |
| prognozowanie | forecasting | przewidywanie przyszłych wartości |
| horyzont prognozy | forecast horizon (fh) | na ile kroków do przodu prognozujemy |
| trend | trend | długoterminowy kierunek zmian |
| sezonowość | seasonality | regularnie powtarzający się wzór |
| dekompozycja | decomposition | rozkład szeregu na składowe |
| stacjonarność | stationarity | stałe właściwości szeregu w czasie |
| autokorelacja | autocorrelation (ACF) | podobieństwo szeregu do jego przeszłych wartości |
| wygładzanie wykładnicze | exponential smoothing | średnia ważona z większą wagą dla nowych danych |
| rosnące okno | expanding window | walidacja: trening zawsze na przeszłości |
| średni bezwzględny błąd procentowy | Mean Absolute Percentage Error (MAPE) | błąd w procentach |

➡️ **Za tydzień:** budujemy własną **aplikację internetową** w Streamlit! 🌐